# Day 24: SQL Basics with SQLite

## Setting Up a Database
SQL is the standard language for querying structured data, and most real-world data actually lives in a relational database, not a CSV — so being able to read, filter, and aggregate with SQL is a core data science skill alongside pandas. `sqlite3` ships with Python's standard library, so no server setup or installation is needed to practice real SQL — `:memory:` creates a temporary database that exists only in RAM for this session, perfect for learning. A `CREATE TABLE` statement defines a table's structure upfront: its column names and data types (`INTEGER`, `TEXT`), plus constraints like `PRIMARY KEY`, which uniquely identifies each row. This cell doesn't print anything — it just creates an empty `employees` table ready to be filled in the next step, and `conn.commit()` saves the change.</cell id="cell-1">


In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')  # ':memory:' = a temporary, in-RAM database (nothing saved to disk)
cursor = conn.cursor()  # the cursor is what actually executes SQL statements and fetches results
cursor.execute('''
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary INTEGER
    )
''')
conn.commit()  # persist the schema change to the database

## Inserting Data
A table with a defined structure but no rows isn't useful yet — `INSERT` is how you add actual data to it. In practice, you'll almost always insert many rows at once rather than one at a time, so `executemany()` is the efficient way to do that: it takes one SQL template with `?` placeholders and a list of tuples, running the insert once per tuple. Using `?` placeholders (instead of building the SQL string by hand with string formatting) is also the safe way to insert data — it avoids SQL injection issues and correctly handles special characters. This cell prints nothing, but afterward the `employees` table holds 4 rows, one per tuple in the list, ready to be queried in the next cell.</cell id="cell-3">


In [ ]:
employees = [
    (1, 'Alice', 'IT', 55000),
    (2, 'Bob', 'HR', 48000),
    (3, 'Charlie', 'IT', 62000),
    (4, 'Diana', 'Sales', 51000)
]
# executemany runs the INSERT once per tuple; ? are placeholders filled in order, safely (no SQL injection)
cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?)', employees)
conn.commit()

## SELECT, WHERE, ORDER BY
These three keywords are the backbone of almost every SQL query you'll ever write. `SELECT` chooses which columns to return (here, just `name` and `salary` instead of every column), `WHERE` filters which rows are included based on a condition (the SQL equivalent of a pandas boolean mask), and `ORDER BY ... DESC` sorts the results, with `DESC` meaning highest-to-lowest (use `ASC`, the default, for lowest-to-highest). This mirrors filtering and sorting a DataFrame in pandas, just expressed in SQL's declarative syntax instead. The output is a list of `(name, salary)` tuples for employees earning more than 50000, sorted from highest salary to lowest.</cell id="cell-5">


In [ ]:
# SELECT specific columns, WHERE filters rows by condition, ORDER BY ... DESC sorts highest salary first
cursor.execute('SELECT name, salary FROM employees WHERE salary > 50000 ORDER BY salary DESC')
print(cursor.fetchall())  # fetchall() retrieves all matching rows as a list of tuples

## Loading Query Results into Pandas
In practice, you'll often want to pull data from a SQL database straight into pandas so you can continue analysis with familiar DataFrame tools. `pd.read_sql_query()` does exactly that in one step: it runs the given SQL query against the connection and returns the results as a DataFrame, with column names taken automatically from the query. This is the standard bridge between "data lives in a database" and "I want to analyze it with pandas" — no manual cursor/fetch loop required. The output is the full `employees` table as a DataFrame, with proper column headers and an integer row index, ready for any pandas operation from earlier in the course.</cell id="cell-7">


In [ ]:
import pandas as pd
# read_sql_query runs the SQL and returns the result directly as a DataFrame
df = pd.read_sql_query('SELECT * FROM employees', conn)
print(df)